# Group 1: Count-Based Models

## Setup and shared preprocessing

Going back to the parent directory to retrieve the current pipeline for building models for later comparison.

In [2]:
import sys
sys.path.append("..")
from src.config import *
from nltk.tokenize import TreebankWordTokenizer
from collections import Counter
import math
import nltk
nltk.download("averaged_perceptron_tagger_eng")
import numpy as np

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\freej\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Loading the training and validation splits. Sanity check is performed in the function as the function coding in the config file checks counts to expected counts.

In [3]:
train_df = load_split('train')
val_df = load_split('validation')


In [4]:
train_df.shape, val_df.shape

((3668, 4), (408, 4))

In [5]:
tokenizer = TreebankWordTokenizer()
train_sentence = train_df['sentence1'].iloc[0]
train_sentence




'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .'

In [6]:
train_sentence_tokens = tokenizer.tokenize(train_sentence)
train_sentence_tokens

['Amrozi',
 'accused',
 'his',
 'brother',
 ',',
 'whom',
 'he',
 'called',
 '``',
 'the',
 'witness',
 '``',
 ',',
 'of',
 'deliberately',
 'distorting',
 'his',
 'evidence',
 '.']

In [7]:
def tokenize_text(text):
    if LOWERCASE:
        text = text.lower()
    return tokenizer.tokenize(text)

train_df['sentence1_tokens'] = train_df['sentence1'].apply(tokenize_text)
train_df['sentence2_tokens'] = train_df['sentence2'].apply(tokenize_text)
train_df[['sentence1_tokens', 'sentence2_tokens']].head()

,sentence1_tokens,sentence2_tokens
0,"[amrozi, accused, his, brother, ,, whom, he, c...","[referring, to, him, as, only, ``, the, witnes..."
1,"[yucaipa, owned, dominick, 's, before, selling...","[yucaipa, bought, dominick, 's, in, 1995, for,..."
2,"[they, had, published, an, advertisement, on, ...","[on, june, 10, ,, the, ship, 's, owners, had, ..."
3,"[around, 0335, gmt, ,, tab, shares, were, up, ...","[tab, shares, jumped, 20, cents, ,, or, 4.6, %..."
4,"[the, stock, rose, $, 2.11, ,, or, about, 11, ...","[pg, &, e, corp., shares, jumped, $, 1.63, or,..."


In [8]:
token_counter = Counter()
for index, row in train_df.iterrows():
    token_counter.update(row['sentence1_tokens'])
    token_counter.update(row['sentence2_tokens'])

token_counter.most_common(10)

[('the', 8477),
 (',', 7978),
 ('.', 7510),
 ('to', 3613),
 ('of', 3420),
 ('in', 3119),
 ('a', 3109),
 ('and', 2879),
 ('``', 2523),
 ('said', 1796)]

In [9]:
vocab = {
    token
    for token, count in token_counter.items()
    if count >= MIN_COUNT
}

len(token_counter), len(vocab)

(13905, 9926)

In [10]:
def apply_vocab(tokens):
    return [
    token if token in vocab else '<UNK>'
    for token in tokens
]

train_df['sentence1_tokens_cleaned'] = train_df['sentence1_tokens'].apply(apply_vocab)
train_df['sentence2_tokens_cleaned'] = train_df['sentence2_tokens'].apply(apply_vocab)

train_df[['sentence1_tokens_cleaned', 'sentence2_tokens_cleaned']].head()

,sentence1_tokens_cleaned,sentence2_tokens_cleaned
0,"[amrozi, accused, his, brother, ,, whom, he, c...","[referring, to, him, as, only, ``, the, witnes..."
1,"[yucaipa, owned, dominick, 's, before, selling...","[yucaipa, bought, dominick, 's, in, 1995, for,..."
2,"[they, had, published, an, advertisement, on, ...","[on, june, 10, ,, the, ship, 's, owners, had, ..."
3,"[around, <UNK>, gmt, ,, tab, shares, were, up,...","[tab, shares, jumped, 20, cents, ,, or, 4.6, %..."
4,"[the, stock, rose, $, 2.11, ,, or, about, 11, ...","[pg, &, e, corp., shares, jumped, $, <UNK>, or..."


In [11]:
uni_paraphrase_token_counter = Counter()
uni_non_paraphrase_token_counter = Counter()

for index, row in train_df.iterrows():
    if row['label'] == 1:
        uni_paraphrase_token_counter.update(row['sentence1_tokens_cleaned'])
        uni_paraphrase_token_counter.update(row['sentence2_tokens_cleaned'])
    else:
        uni_non_paraphrase_token_counter.update(row['sentence1_tokens_cleaned'])
        uni_non_paraphrase_token_counter.update(row['sentence2_tokens_cleaned'])

uni_paraphrase_token_total = sum(uni_paraphrase_token_counter.values())
uni_non_paraphrase_token_total = sum(uni_non_paraphrase_token_counter.values())

uni_paraphrase_token_total, uni_non_paraphrase_token_total

(111666, 49395)

Cleaning validation set for validating after training.

In [12]:
val_df['sentence1_tokens'] = val_df['sentence1'].apply(tokenize_text)
val_df['sentence2_tokens'] = val_df['sentence2'].apply(tokenize_text)
val_df[['sentence1_tokens', 'sentence2_tokens']].head()

,sentence1_tokens,sentence2_tokens
0,"[he, said, the, foodservice, pie, business, do...","[``, the, foodservice, pie, business, does, no..."
1,"[magnarelli, said, racicot, hated, the, iraqi,...","[his, wife, said, he, was, ``, 100, percent, b..."
2,"[the, dollar, was, at, 116.92, yen, against, t...","[the, dollar, was, at, 116.78, yen, jpy, =, ,,..."
3,"[the, afl-cio, is, waiting, until, october, to...","[the, afl-cio, announced, wednesday, that, it,..."
4,"[no, dates, have, been, set, for, the, civil, ...","[no, dates, have, been, set, for, the, crimina..."


In [13]:
val_token_counter = Counter()

for index, row in val_df.iterrows():
    val_token_counter.update(row['sentence1_tokens'])
    val_token_counter.update(row['sentence2_tokens'])

val_token_counter.most_common(10)

[('the', 980),
 ('.', 832),
 (',', 812),
 ('to', 411),
 ('a', 350),
 ('of', 348),
 ('in', 344),
 ('and', 296),
 ('``', 275),
 ('said', 227)]

In [14]:
val_df['sentence1_tokens_cleaned'] = val_df['sentence1_tokens'].apply(apply_vocab)
val_df['sentence2_tokens_cleaned'] = val_df['sentence2_tokens'].apply(apply_vocab)

val_df[['sentence1_tokens_cleaned', 'sentence2_tokens_cleaned']].head()

,sentence1_tokens_cleaned,sentence2_tokens_cleaned
0,"[he, said, the, foodservice, <UNK>, business, ...","[``, the, foodservice, <UNK>, business, does, ..."
1,"[<UNK>, said, racicot, <UNK>, the, iraqi, regi...","[his, wife, said, he, was, ``, 100, percent, b..."
2,"[the, dollar, was, at, <UNK>, yen, against, th...","[the, dollar, was, at, <UNK>, yen, jpy, =, ,, ..."
3,"[the, <UNK>, is, waiting, until, october, to, ...","[the, <UNK>, announced, wednesday, that, it, w..."
4,"[no, dates, have, been, set, for, the, civil, ...","[no, dates, have, been, set, for, the, crimina..."


In [15]:
"<UNK>" in vocab

False

## N-gram models

# Unigram

Using Laplace smoothing for unknown tokens. One is added in the denominator for the unknown token that is not in the vocabulary.

In [16]:
uni_predictions = []
uni_paraphrase_log_likelihoods = []
uni_non_paraphrase_log_likelihoods = []

for index, row in val_df.iterrows():
    paraphrase_score = 0
    non_paraphrase_score = 0

    tokens = row['sentence1_tokens_cleaned'] + row['sentence2_tokens_cleaned']

    for token in tokens:
        paraphrase_score += math.log((uni_paraphrase_token_counter.get(token, 0) + 1) / (
            uni_paraphrase_token_total + len(vocab) + 1
        ))
        non_paraphrase_score += math.log((uni_non_paraphrase_token_counter.get(token, 0) + 1) / (
            uni_non_paraphrase_token_total + len(vocab) + 1
        ))

    uni_paraphrase_log_likelihoods.append(paraphrase_score)
    uni_non_paraphrase_log_likelihoods.append(non_paraphrase_score)

    if paraphrase_score > non_paraphrase_score:
        uni_predictions.append(1)
    else:
        uni_predictions.append(0)

len(uni_paraphrase_log_likelihoods), len(uni_non_paraphrase_log_likelihoods), len(uni_predictions)

(408, 408, 408)

Checking the outputs.

In [17]:
uni_paraphrase_log_likelihoods[:10], uni_non_paraphrase_log_likelihoods[:10], uni_predictions[:10]

([-201.63007084900988,
  -289.6559048320029,
  -342.91514354667896,
  -219.3106812740883,
  -208.99178801029208,
  -274.62583301393903,
  -256.27594133409093,
  -314.94626999970467,
  -202.53524500404617,
  -316.97788823291853],
 [-207.15607865033016,
  -293.8722466883388,
  -331.03233553386366,
  -222.55887232832188,
  -209.01261391296734,
  -287.6018851154298,
  -259.12553178588126,
  -322.95587357859125,
  -198.22082197069977,
  -336.42420171671324],
 [1, 1, 0, 1, 1, 1, 1, 1, 0, 1])

In [18]:
Counter(uni_predictions)

Counter({1: 267, 0: 141})

In [19]:
uni_log_likelihood_differences = []

for i in range(len(uni_paraphrase_log_likelihoods)):
    uni_log_likelihood_differences.append(uni_paraphrase_log_likelihoods[i] - uni_non_paraphrase_log_likelihoods[i])

uni_prob_paraphrases = []

for diff in uni_log_likelihood_differences:
    uni_prob_paraphrases.append(1 / (1 + math.exp(-diff)))

[(uni_prob_paraphrases[i], uni_predictions[i]) for i in range(10)]

[(0.996033938524896, 1),
 (0.9854619598947345, 1),
 (6.908106914104177e-06, 0),
 (0.9626080563076677, 1),
 (0.5052062874983609, 1),
 (0.9999976848924568, 1),
 (0.9452975088552688, 1),
 (0.9996678539513176, 1),
 (0.013197753527898422, 0),
 (0.9999999964143049, 1)]

In [20]:
uni_prediction_df = pd.DataFrame({
    'idx': val_df['idx'],
    'label': val_df['label'],
    'prob_paraphrase': uni_prob_paraphrases,
    'pred': uni_predictions
})

uni_prediction_df = uni_prediction_df[PRED_COLUMNS]

uni_prediction_df.head()

,idx,label,prob_paraphrase,pred
0,9,1,0.996034,1
1,18,0,0.985462,1
2,25,0,0.000007,0
3,32,1,0.962608,1
4,33,0,0.505206,1


In [21]:
val_df[["idx", "label"]].head()

,idx,label
0,9,1
1,18,0
2,25,0
3,32,1
4,33,0


# Unigram results

In [22]:
uni_results = evaluate(val_df['label'], uni_predictions)
uni_results

{'accuracy': 0.6372549019607843, 'f1': 0.7289377289377289}

# Bigram


Same idea but using the previous word for added context. "Training" the model to use the counts when we run it on the validation set.

In [23]:
bi_paraphrase_token_counter = Counter()
bi_non_paraphrase_token_counter = Counter()

bi_paraphrase_context_counter = Counter()
bi_non_paraphrase_context_counter = Counter()

for index, row in train_df.iterrows():
    tokens = (
        row['sentence1_tokens_cleaned']
        + ['<SEP>']
        + row['sentence2_tokens_cleaned']
    )

    for i in range(len(tokens) - 1):
        previous_word = tokens[i]
        current_word = tokens[i + 1]
        bigram = (previous_word, current_word)

        if row['label'] == 1:
            bi_paraphrase_token_counter[bigram] += 1
            bi_paraphrase_context_counter[bigram] += 1
        else:
            bi_non_paraphrase_token_counter[bigram] += 1
            bi_non_paraphrase_context_counter[bigram] += 1

bi_paraphrase_token_total = sum(bi_paraphrase_token_counter.values())
bi_non_paraphrase_token_total = sum(bi_non_paraphrase_token_counter.values())

bi_paraphrase_token_total, bi_non_paraphrase_token_total

(111666, 49395)

In [24]:
bi_paraphrase_token_counter.most_common(5)

[(('.', '<SEP>'), 2340),
 ((',', '``'), 563),
 (('of', 'the'), 525),
 (('in', 'the'), 522),
 (('<SEP>', 'the'), 451)]

In [25]:
bi_predictions = []
bi_paraphrase_log_likelihoods = []
bi_non_paraphrase_log_likelihoods = []

for index, row in val_df.iterrows():
    paraphrase_score = 0
    non_paraphrase_score = 0

    tokens = (
        row['sentence1_tokens_cleaned']
        + ['<SEP>']
        + row['sentence2_tokens_cleaned']
    )

    for i in range(len(tokens) - 1):
        previous_word = tokens[i]
        current_word = tokens[i + 1]
        bigram = (previous_word, current_word)

        paraphrase_score += math.log((bi_paraphrase_token_counter.get(bigram, 0) + 1) / (
            bi_paraphrase_context_counter.get(previous_word, 0) + len(vocab) + 2
        ))

        non_paraphrase_score += math.log((bi_non_paraphrase_token_counter.get(bigram, 0) + 1) / (
            bi_non_paraphrase_context_counter.get(previous_word, 0) + len(vocab) + 2
        ))

    bi_paraphrase_log_likelihoods.append(paraphrase_score)
    bi_non_paraphrase_log_likelihoods.append(non_paraphrase_score)

    if paraphrase_score > non_paraphrase_score:
        bi_predictions.append(1)
    else:
        bi_predictions.append(0)

len(bi_paraphrase_log_likelihoods), len(bi_non_paraphrase_log_likelihoods), len(bi_predictions)

(408, 408, 408)

Checking outputs

In [26]:
bi_paraphrase_log_likelihoods[:10], bi_non_paraphrase_log_likelihoods[:10], bi_predictions[:10]

([-221.86466919299858,
  -355.63882492808716,
  -401.96410066151526,
  -258.1284338228105,
  -246.4615653070654,
  -304.599467547457,
  -307.65102414315277,
  -377.0438711815213,
  -248.9492484732696,
  -336.716236939877],
 [-234.97500763544724,
  -372.8721090181574,
  -395.04653572569606,
  -271.6569317160478,
  -240.44843091060724,
  -330.4489315100413,
  -325.14578332385724,
  -396.790173662177,
  -252.68809159909168,
  -393.1165141972781],
 [1, 1, 0, 1, 0, 1, 1, 1, 1, 1])

In [27]:
Counter(bi_predictions)

Counter({1: 381, 0: 27})

In [28]:
bi_log_likelihood_differences = []

for i in range(len(bi_paraphrase_log_likelihoods)):
    bi_log_likelihood_differences.append(bi_paraphrase_log_likelihoods[i] - bi_non_paraphrase_log_likelihoods[i])

bi_prob_paraphrases = []

for diff in bi_log_likelihood_differences:
    bi_prob_paraphrases.append(1 / (1 + math.exp(-diff)))

[(bi_prob_paraphrases[i], bi_predictions[i]) for i in range(10)]

[(0.9999979758090471, 1),
 (0.9999999672146518, 1),
 (0.000989258699048838, 0),
 (0.999998667560687, 1),
 (0.0024404378305825526, 0),
 (0.999999999994061, 1),
 (0.9999999747580668, 1),
 (0.9999999973436224, 1),
 (0.9767708273401674, 1),
 (1.0, 1)]

In [29]:
bi_prediction_df = pd.DataFrame({
    'idx': val_df['idx'],
    'label': val_df['label'],
    'prob_paraphrase': bi_prob_paraphrases,
    'pred': bi_predictions
})

bi_prediction_df = bi_prediction_df[PRED_COLUMNS]

bi_prediction_df.head()

,idx,label,prob_paraphrase,pred
0,9,1,0.999998,1
1,18,0,1.000000,1
2,25,0,0.000989,0
3,32,1,0.999999,1
4,33,0,0.002440,0


Slightly better than unigram but also the dataset favors paraphrases and the model overwhelmingly guesses paraphrase.

## Bigram results

In [30]:
bi_results = evaluate(val_df["label"], bi_predictions)
bi_results

{'accuracy': 0.7058823529411765, 'f1': 0.8181818181818182}

## HMM model

Creating the paraphrase and non datasets to build the HMM on and then use vertibi for our classification.

In [31]:
from nltk.tag.hmm import HiddenMarkovModelTrainer
from nltk.probability import LaplaceProbDist

In [32]:
paraphrase_train = train_df[train_df["label"] == 1]
non_paraphrase_train = train_df[train_df["label"] == 0]

nltk.pos_tag(paraphrase_train['sentence1_tokens_cleaned'].iloc[0])

[('amrozi', 'NN'),
 ('accused', 'VBD'),
 ('his', 'PRP$'),
 ('brother', 'NN'),
 (',', ','),
 ('whom', 'WP'),
 ('he', 'PRP'),
 ('called', 'VBD'),
 ('``', '``'),
 ('the', 'DT'),
 ('witness', 'NN'),
 ('``', '``'),
 (',', ','),
 ('of', 'IN'),
 ('deliberately', 'RB'),
 ('distorting', 'VBG'),
 ('his', 'PRP$'),
 ('evidence', 'NN'),
 ('.', '.')]

In [33]:
paraphrase_sentence1_pos = []
paraphrase_sentence2_pos = []

for index, row in paraphrase_train.iterrows():
    sentence1_pos = nltk.pos_tag(row['sentence1_tokens_cleaned'])
    sentence2_pos = nltk.pos_tag(row['sentence2_tokens_cleaned'])

    paraphrase_sentence1_pos.append(sentence1_pos)
    paraphrase_sentence2_pos.append(sentence2_pos)

paraphrase_train['paraphrase_sentence1_pos'] = paraphrase_sentence1_pos
paraphrase_train['paraphrase_sentence2_pos'] = paraphrase_sentence2_pos

len(paraphrase_sentence1_pos), len(paraphrase_sentence2_pos), len(paraphrase_train)

(2474, 2474, 2474)

In [34]:
non_paraphrase_sentence1_pos = []
non_paraphrase_sentence2_pos = []

for index, row in non_paraphrase_train.iterrows():
    sentence1_pos = nltk.pos_tag(row['sentence1_tokens_cleaned'])
    sentence2_pos = nltk.pos_tag(row['sentence2_tokens_cleaned'])

    non_paraphrase_sentence1_pos.append(sentence1_pos)
    non_paraphrase_sentence2_pos.append(sentence2_pos)

non_paraphrase_train['non_paraphrase_sentence1_pos'] = non_paraphrase_sentence1_pos
non_paraphrase_train['non_paraphrase_sentence2_pos'] = non_paraphrase_sentence2_pos

len(non_paraphrase_sentence1_pos), len(non_paraphrase_sentence2_pos), len(non_paraphrase_train)

(1194, 1194, 1194)

In [35]:
paraphrase_sequences = (
    paraphrase_train["paraphrase_sentence1_pos"].tolist()
    + paraphrase_train["paraphrase_sentence2_pos"].tolist()
)

non_paraphrase_sequences = (
    non_paraphrase_train["non_paraphrase_sentence1_pos"].tolist()
    + non_paraphrase_train["non_paraphrase_sentence2_pos"].tolist()
)

In [36]:
laplace_estimator = lambda fd, bins: LaplaceProbDist(fd, bins)

paraphrase_trainer = HiddenMarkovModelTrainer()
non_paraphrase_trainer = HiddenMarkovModelTrainer()

paraphrase_hmm = paraphrase_trainer.train_supervised(
    paraphrase_sequences,
    estimator=laplace_estimator
)

non_paraphrase_hmm = non_paraphrase_trainer.train_supervised(
    non_paraphrase_sequences,
    estimator=laplace_estimator
)

In [37]:
hmm_predictions = []
hmm_paraphrase_scores = []
hmm_non_paraphrase_scores = []

for _, row in val_df.iterrows():
    sentence1 = row["sentence1_tokens_cleaned"]
    sentence2 = row["sentence2_tokens_cleaned"]

    # Paraphrase HMM
    para_tagged_s1 = paraphrase_hmm.tag(sentence1)
    para_tagged_s2 = paraphrase_hmm.tag(sentence2)

    para_score = (
        paraphrase_hmm.log_probability(para_tagged_s1)
        + paraphrase_hmm.log_probability(para_tagged_s2)
    )

    # Non-paraphrase HMM
    nonpara_tagged_s1 = non_paraphrase_hmm.tag(sentence1)
    nonpara_tagged_s2 = non_paraphrase_hmm.tag(sentence2)

    nonpara_score = (
        non_paraphrase_hmm.log_probability(nonpara_tagged_s1)
        + non_paraphrase_hmm.log_probability(nonpara_tagged_s2)
    )

    prediction = 1 if para_score > nonpara_score else 0

    hmm_paraphrase_scores.append(para_score)
    hmm_non_paraphrase_scores.append(nonpara_score)
    hmm_predictions.append(prediction)

In [38]:
Counter(hmm_predictions)

Counter({1: 314, 0: 94})

In [39]:
hmm_prediction_df = pd.DataFrame({
    "idx": val_df["idx"],
    "label": val_df["label"],
    "prob_paraphrase": [
        1 / (1 + np.exp(-(p - n)))
        for p, n in zip(hmm_paraphrase_scores, hmm_non_paraphrase_scores)
    ],
    "pred": hmm_predictions
})

hmm_prediction_df = hmm_prediction_df[PRED_COLUMNS]

hmm_prediction_df.head()

,idx,label,prob_paraphrase,pred
0,9,1,9.999997e-01,1
1,18,0,9.998932e-01,1
2,25,0,3.601728e-11,0
3,32,1,9.999864e-01,1
4,33,0,9.998547e-01,1


## HMM evaluation

In [40]:
hmm_results = evaluate(
    hmm_prediction_df["label"],
    hmm_prediction_df["pred"]
)

hmm_results

{'accuracy': 0.6740196078431373, 'f1': 0.7757166947723441}

## Comparison

In [41]:
group1_results = pd.DataFrame([
    {
        "model": "Unigram",
        "accuracy": uni_results["accuracy"],
        "f1": uni_results["f1"]
    },
    {
        "model": "Bigram",
        "accuracy": bi_results["accuracy"],
        "f1": bi_results["f1"]
    },
    {
        "model": "HMM",
        "accuracy": hmm_results["accuracy"],
        "f1": hmm_results["f1"]
    }
])

group1_results

,model,accuracy,f1
0,Unigram,0.637255,0.728938
1,Bigram,0.705882,0.818182
2,HMM,0.674020,0.775717


# Raw vs clean

Creating raw bigram counters and the vocabulary

In [42]:
raw_bi_paraphrase_counter = Counter()
raw_bi_non_paraphrase_counter = Counter()

raw_bi_paraphrase_context = Counter()
raw_bi_non_paraphrase_context = Counter()

raw_vocab = set()

for _, row in train_df.iterrows():
    raw_vocab.update(row["sentence1_tokens"])
    raw_vocab.update(row["sentence2_tokens"])

raw_bigram_vocab_size = len(raw_vocab) + 1   # +1 for <SEP>

Counting raw bigrams

In [43]:
for _, row in train_df.iterrows():

    sequence = (
        row["sentence1_tokens"]
        + ["<SEP>"]
        + row["sentence2_tokens"]
    )

    for i in range(len(sequence) - 1):
        previous_word = sequence[i]
        current_word = sequence[i + 1]

        if row["label"] == 1:
            raw_bi_paraphrase_counter[(previous_word, current_word)] += 1
            raw_bi_paraphrase_context[previous_word] += 1

        else:
            raw_bi_non_paraphrase_counter[(previous_word, current_word)] += 1
            raw_bi_non_paraphrase_context[previous_word] += 1

Scoring validation pairs

In [44]:
raw_bi_predictions = []
raw_bi_paraphrase_scores = []
raw_bi_non_paraphrase_scores = []

for _, row in val_df.iterrows():

    sequence = (
        row["sentence1_tokens"]
        + ["<SEP>"]
        + row["sentence2_tokens"]
    )

    paraphrase_score = 0
    non_paraphrase_score = 0

    for i in range(len(sequence) - 1):
        previous_word = sequence[i]
        current_word = sequence[i + 1]

        paraphrase_prob = (
            raw_bi_paraphrase_counter.get(
                (previous_word, current_word), 0
            ) + 1
        ) / (
            raw_bi_paraphrase_context.get(previous_word, 0)
            + raw_bigram_vocab_size
        )

        non_paraphrase_prob = (
            raw_bi_non_paraphrase_counter.get(
                (previous_word, current_word), 0
            ) + 1
        ) / (
            raw_bi_non_paraphrase_context.get(previous_word, 0)
            + raw_bigram_vocab_size
        )

        paraphrase_score += math.log(paraphrase_prob)
        non_paraphrase_score += math.log(non_paraphrase_prob)

    prediction = 1 if paraphrase_score > non_paraphrase_score else 0

    raw_bi_paraphrase_scores.append(paraphrase_score)
    raw_bi_non_paraphrase_scores.append(non_paraphrase_score)
    raw_bi_predictions.append(prediction)

In [45]:
Counter(raw_bi_predictions)

Counter({1: 377, 0: 31})

# Raw vs clean results

In [46]:
raw_bi_results = evaluate(
    val_df["label"],
    raw_bi_predictions
)

raw_bi_results

{'accuracy': 0.7058823529411765, 'f1': 0.8170731707317073}

In [47]:
bigram_cleaning_comparison = pd.DataFrame([
    {
        "version": "Raw Bigram",
        "accuracy": raw_bi_results["accuracy"],
        "f1": raw_bi_results["f1"]
    },
    {
        "version": "Cleaned Bigram",
        "accuracy": bi_results["accuracy"],
        "f1": bi_results["f1"]
    }
])

bigram_cleaning_comparison

,version,accuracy,f1
0,Raw Bigram,0.705882,0.817073
1,Cleaned Bigram,0.705882,0.818182
